Dataset audit

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/raw/dataset-2.csv')
df.info()
display(df.head())

print("=================================================================================")
print(df.describe())
print("=================================================================================")
print(df.columns)
print("=================================null================================================")
print(df.isnull().sum())
print("=================================nan================================================")
print(df.isna().sum())
print("===================================dup============================================")
print(df.duplicated().sum())
print("===================================shape============================================")
print(df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_condition          

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


       Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
count         45593.000000          45593.000000                45593.000000   
mean             17.017729             70.231332                   17.465186   
std               8.185109             22.883647                    7.335122   
min             -30.905562            -88.366217                    0.010000   
25%              12.933284             73.170000                   12.988453   
50%              18.546947             75.898497                   18.633934   
75%              22.728163             78.044095                   22.785049   
max              30.914057             88.433452                   31.054057   

       Delivery_location_longitude  Vehicle_condition  
count                 45593.000000       45593.000000  
mean                     70.845702           1.023359  
std                      21.118812           0.839065  
min                       0.010000           0.000000  

identify the target

the target is the delivery time in minutes its the most important feature to predict , we can rely on it for prediction of the delivery time , its already integer and have a good name that indicate that the values are by minute .

In [2]:
target = df['Time_taken(min)']
print(target.head(3))
print(target.dtype)

data_dictionary = pd.DataFrame([
    ("ID", "Order identifier", "None", "Exclude: identifier"),
    ("Delivery_person_ID", "Courier identifier", "None", "Exclude: identifier"),
    ("Delivery_person_Age", "Courier age", "Years", "Predictor"),
    ("Delivery_person_Ratings", "Courier rating", "Score", "Predictor"),
    ("Restaurant_latitude", "Restaurant latitude", "Degrees", "Derive distance"),
    ("Restaurant_longitude", "Restaurant longitude", "Degrees", "Derive distance"),
    ("Delivery_location_latitude", "Customer latitude", "Degrees", "Derive distance"),
    ("Delivery_location_longitude", "Customer longitude", "Degrees", "Derive distance"),
    ("Order_Date", "Order date", "Date", "Derive date features"),
    ("Time_Orderd", "Order placement time", "Clock time", "Derive time features"),
    ("Time_Order_picked", "Courier pickup time", "Clock time", "Check availability/leakage"),
    ("Weatherconditions", "Weather condition", "Category", "Predictor"),
    ("Road_traffic_density", "Traffic level", "Category", "Predictor"),
    ("Vehicle_condition", "Vehicle condition level", "Ordinal", "Predictor"),
    ("Type_of_order", "Order category", "Category", "Predictor"),
    ("Type_of_vehicle", "Courier vehicle type", "Category", "Predictor"),
    ("multiple_deliveries", "Concurrent delivery count", "Count", "Predictor"),
    ("Festival", "Festival-day indicator", "Category", "Predictor"),
    ("City", "City category", "Category", "Predictor"),
    ("Time_taken(min)", "Observed delivery duration", "Minutes", "Target"),
], columns=["column", "meaning", "unit", "modeling_decision"])
data_dictionary.insert(2, "dtype", data_dictionary["column"].map(df.dtypes.astype(str)))

display(data_dictionary)

0    (min) 24
1    (min) 33
2    (min) 26
Name: Time_taken(min), dtype: str
str


,column,meaning,dtype,unit,modeling_decision
0,ID,Order identifier,str,None,Exclude: identifier
1,Delivery_person_ID,Courier identifier,str,None,Exclude: identifier
2,Delivery_person_Age,Courier age,str,Years,Predictor
3,Delivery_person_Ratings,Courier rating,str,Score,Predictor
4,Restaurant_latitude,Restaurant latitude,float64,Degrees,Derive distance
5,Restaurant_longitude,Restaurant longitude,float64,Degrees,Derive distance
6,Delivery_location_latitude,Customer latitude,float64,Degrees,Derive distance
7,Delivery_location_longitude,Customer longitude,float64,Degrees,Derive distance
8,Order_Date,Order date,str,Date,Derive date features
9,Time_Orderd,Order placement time,str,Clock time,Derive time features


cleaning data setup

In [1]:
from src.data import (
load_raw_data,
clean_delivery_data,
create_train_test_split,
)
raw_df = load_raw_data()


cleaned_df = clean_delivery_data(raw_df)

print("raw shape:", raw_df.shape)
print("cleaned shape:", cleaned_df.shape)
display(cleaned_df.head(3))
print(cleaned_df.isna().sum())
print("exact duplicates", raw_df.duplicated().sum())



raw shape: (45593, 20)
cleaned shape: (45593, 10)


,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_vehicle,Time_taken(min)
0,22.745049,75.892471,22.765049,75.912471,0 days 11:45:00,Sunny,2,2,Motorcycle,24
1,12.913041,77.683237,13.043041,77.813237,0 days 19:50:00,Stormy,3,2,Scooter,33
2,12.914264,77.678400,12.924264,77.688400,0 days 08:45:00,Sandstorms,0,0,Motorcycle,26


Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_vehicle                0
Time_taken(min)                0
dtype: int64
exact duplicates 0


Visualisation

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.scatter(cleaned_df["Preparation_Time_min"], cleaned_df["Delivery_Time_min"])
sns.regplot(
    data=cleaned_df,
    x = "Preparation_Time_min",
    y = "Delivery_Time_min",
    scatter_kws={"alpha": 0.5},
    line_kws={"color": "red"}
)

plt.title("does delivery related to deliveryt time    ")
plt.show()
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
      cleaned_df["Delivery_Time_min"],
      bins=40,
      edgecolor="black"
  )

ax.set_title("Distrubition")
ax.set_xlabel("Delivery time")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

KeyError: 'Preparation_Time_min'

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

ax.boxplot(
  cleaned_df["Delivery_Time_min"].dropna(),
  vert=False
)

ax.set_title("Delivery Time Spread and Potential Outliers")
ax.set_xlabel("Delivery time (minutes)")

plt.tight_layout()
plt.show()




analyse descriptive

In [ ]:
xtrain , xtest, ytrain, ytest = create_train_test_split(cleaned_df)
train_eda = xtrain.copy()
train_eda["Delivery_Time_min"] = ytrain
print(train_eda.columns)
print(train_eda.describe())
print(train_eda["Weather"].value_counts(), train_eda["Weather"].value_counts(normalize=True).mul(100))
print(train_eda["Traffic_Level"].value_counts(), train_eda["Traffic_Level"].value_counts(normalize=True).mul(100))
print(train_eda["Vehicle_Type"].value_counts(),train_eda["Vehicle_Type"].value_counts(normalize=True).mul(100))


plot the target distribution and report its median , spread and long tail

In [ ]:
ax.hist(
      ytrain,
      bins=40,
      edgecolor="black"
  )

ax.set_title("Distrubition")
ax.set_xlabel("Delivery time")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()